In [32]:
# Install requirements
!pip --quiet install streamlit streamviz tensorflow keras numpy pandas watchdog joblib

In [33]:
!pip install scikit-learn

In [36]:
%%writefile ._app.py
import streamlit as st
import tensorflow as tf
import numpy as np
import pickle
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from keras.preprocessing.sequence import pad_sequences

# Load model
model = tf.keras.models.load_model("spam_detector_model.h5")

# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

max_len = 478  # same as in training

# Use sklearn's stopwords
stop_words = set(ENGLISH_STOP_WORDS)

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    words = text.split()
    words = [word for word in words if word not in stop_words]  # remove stopwords
    return ' '.join(words)

# Streamlit layout
st.set_page_config(page_title="🔍 Email Spam Detector", layout="centered")
st.markdown("<h1 style='text-align: center; color: #333;'>🔍 Email Spam Detector</h1>", unsafe_allow_html=True)
st.write("")

# Columns layout
col1, col2 = st.columns([1, 2])

with col2:
    email_input = st.text_area("Write your email content below 👇", height=300)
    run = st.button("Run", use_container_width=True)

if run and email_input.strip() != "":
    # Preprocess the input
    cleaned = clean_text(email_input)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = tf.convert_to_tensor(
        pad_sequences(seq, maxlen=max_len, padding='post')
    )

    # Predict
    prediction = model.predict(padded)[0][0]
    spam_score = round(prediction * 100, 2)

    # Label based on threshold
    is_spam = prediction >= 0.8
    label = "🚫 SPAM" if is_spam else "✅ NOT SPAM"

    with col1:
        st.markdown("### Prediction")
        st.write(f"**Result:** {label}")
        st.write(f"**Spam Probability:** {spam_score}%")


Overwriting ._app.py


In [37]:
! echo "" | streamlit run ._app.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.159:8501

2025-04-23 15:26:59.927344: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step
^C
